Trabalho de buscas - Solução do quebra-cabeças deslizante                                                       Estrutura base do quebra-cabeças 3x3:



In [1]:
import time
from collections import deque
import heapq
import copy

# Estado objetivo do 8-puzzle (Matriz 3x3)
GOAL_STATE = [[1, 2, 3],
              [4, 5, 6],
              [7, 8, 0]]

# Mapeamento de movimentos (Linha, Coluna)
MOVES = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}

class Node:
    def __init__(self, state, parent=None, move=None, depth=0, cost=0):
        self.state = state
        self.parent = parent
        self.move = move
        self.depth = depth
        self.cost = cost # Usado para o A* (f(n) = g(n) + h(n))

    # Necessário para a fila de prioridade desempatar nós com mesmo custo
    def __lt__(self, other):
        return self.cost < other.cost

    # Encontra a posição do espaço vazio (0)
    def get_blank_pos(self):
        for i in range(3):
            for j in range(3):
                if self.state[i][j] == 0:
                    return i, j
        return -1, -1

    # Gera os estados filhos a partir dos movimentos válidos
    def generate_children(self):
        children = []
        x, y = self.get_blank_pos()

        for move, (dx, dy) in MOVES.items():
            new_x, new_y = x + dx, y + dy

            # Verifica se o movimento não sai do tabuleiro 3x3
            if 0 <= new_x < 3 and 0 <= new_y < 3:
                new_state = copy.deepcopy(self.state)
                # Troca o espaço vazio (0) com a peça adjacente
                new_state[x][y], new_state[new_x][new_y] = new_state[new_x][new_y], new_state[x][y]

                child_node = Node(new_state, parent=self, move=move, depth=self.depth + 1)
                children.append(child_node)
        return children

    # Reconstrói o caminho do estado inicial até a solução
    def get_path(self):
        path = []
        current = self
        while current.parent:
            path.append(current.move)
            current = current.parent
        return path[::-1] # Inverte a lista para ficar do início ao fim

    # Converte o estado para tupla para podermos usar em um Set (hashável)
    def state_tuple(self):
        return tuple(tuple(row) for row in self.state)

Buscas Cegas (BFS, DFS, DLS, IDS)

In [2]:
def is_goal(state):
    return state == GOAL_STATE

def bfs(start_state):
    start_node = Node(start_state)
    if is_goal(start_node.state): return start_node.get_path(), 0

    frontier = deque([start_node]) # Fila para o BFS
    explored = set()
    nodes_expanded = 0

    while frontier:
        node = frontier.popleft()
        explored.add(node.state_tuple())
        nodes_expanded += 1

        for child in node.generate_children():
            if child.state_tuple() not in explored and all(child.state_tuple() != n.state_tuple() for n in frontier):
                if is_goal(child.state):
                    return child.get_path(), nodes_expanded
                frontier.append(child)
    return None, nodes_expanded

def dfs(start_state):
    start_node = Node(start_state)
    frontier = [start_node] # Pilha para o DFS
    explored = set()
    nodes_expanded = 0

    while frontier:
        node = frontier.pop()
        nodes_expanded += 1

        if is_goal(node.state):
            return node.get_path(), nodes_expanded

        explored.add(node.state_tuple())

        # Adiciona filhos na pilha de forma reversa para manter ordem padrão
        for child in reversed(node.generate_children()):
            if child.state_tuple() not in explored:
                frontier.append(child)
    return None, nodes_expanded

def dls(start_state, limit):
    start_node = Node(start_state)
    frontier = [start_node]
    explored = {} # Guarda o estado e a profundidade em que foi achado
    nodes_expanded = 0

    while frontier:
        node = frontier.pop()
        nodes_expanded += 1

        if is_goal(node.state):
            return node.get_path(), nodes_expanded

        explored[node.state_tuple()] = node.depth

        if node.depth < limit:
            for child in reversed(node.generate_children()):
                state_tup = child.state_tuple()
                # Só explora se não visitou ou se achou um caminho mais curto para o mesmo estado
                if state_tup not in explored or child.depth < explored[state_tup]:
                    frontier.append(child)

    return "CUTOFF", nodes_expanded

def ids(start_state):
    depth = 0
    total_expanded = 0
    while True:
        result, expanded = dls(start_state, depth)
        total_expanded += expanded
        if result != "CUTOFF":
            return result, total_expanded
        depth += 1

Heurísticas e Busca com Informação (A*)

In [3]:
# Heurística 1: Peças Fora do Lugar (Misplaced Tiles / Hamming)
def h1_misplaced_tiles(state):
    count = 0
    for i in range(3):
        for j in range(3):
            if state[i][j] != 0 and state[i][j] != GOAL_STATE[i][j]:
                count += 1
    return count

# Heurística 2: Distância de Manhattan
def h2_manhattan_distance(state):
    dist = 0
    for i in range(3):
        for j in range(3):
            val = state[i][j]
            if val != 0:
                # Encontra a posição correta do valor no GOAL_STATE
                target_i = (val - 1) // 3
                target_j = (val - 1) % 3
                dist += abs(i - target_i) + abs(j - target_j)
    return dist

def a_star(start_state, heuristic_func):
    start_node = Node(start_state)
    start_node.cost = start_node.depth + heuristic_func(start_node.state)

    frontier = []
    heapq.heappush(frontier, start_node) # Fila de prioridade

    # Dicionário para guardar o menor custo g(n) encontrado para cada estado
    g_costs = {start_node.state_tuple(): start_node.depth}
    nodes_expanded = 0

    while frontier:
        node = heapq.heappop(frontier)
        nodes_expanded += 1

        if is_goal(node.state):
            return node.get_path(), nodes_expanded

        for child in node.generate_children():
            g_cost = child.depth
            state_tup = child.state_tuple()

            # Se achamos um caminho melhor (ou novo) para este estado
            if state_tup not in g_costs or g_cost < g_costs[state_tup]:
                g_costs[state_tup] = g_cost
                child.cost = g_cost + heuristic_func(child.state)
                heapq.heappush(frontier, child)

    return None, nodes_expanded

Execução:

In [4]:
import time

# --- Função de Teste e Coleta ---
resultados = []

def run_and_collect(algorithm_name, func, *args):
    print(f"Executando {algorithm_name}...")
    start_time = time.time()

    # Executa o algoritmo
    resultado = func(*args)

    end_time = time.time()
    time_taken = end_time - start_time

    # Extrai o caminho e os nós expandidos dependendo do retorno
    if isinstance(resultado, tuple):
        path, nodes_expanded = resultado
    else:
        path, nodes_expanded = None, 0

    # Verifica se encontrou solução, falhou ou atingiu limite (CUTOFF do DLS)
    if isinstance(path, list):
        num_movimentos = len(path)
    elif path == "CUTOFF":
        num_movimentos = "LIMITE"
    else:
        num_movimentos = "FALHA"

    resultados.append({
        "Algoritmo": algorithm_name,
        "Movimentos": num_movimentos,
        "Tempo (s)": time_taken,
        "Nós Expandidos": nodes_expanded
    })

# --- Estado Inicial de Teste ---
# Dica: Alterem esta matriz para testar casos mais fáceis ou mais difíceis.
embaralhado = [
    [1, 2, 3],
    [5, 0, 6],
    [4, 7, 8]
]

print("Iniciando bateria de testes...\n")

# --- Execução dos Algoritmos ---
run_and_collect("Busca em Largura (BFS)", bfs, embaralhado)
run_and_collect("Busca em Prof. (DFS)", dfs, embaralhado)
run_and_collect("Aprofundamento Iter. (IDS)", ids, embaralhado)
run_and_collect("A* (H1: Peças Fora do Lugar)", a_star, embaralhado, h1_misplaced_tiles)
run_and_collect("A* (H2: Manhattan Distance)", a_star, embaralhado, h2_manhattan_distance)

# --- Impressão da Tabela Comparativa ---
print("\n" + "="*75)
print(f"{'RESULTADOS DA COMPARAÇÃO':^75}")
print("="*75)
print(f"{'Algoritmo':<30} | {'Movimentos':<12} | {'Tempo (s)':<12} | {'Nós Expandidos'}")
print("-" * 75)

for res in resultados:
    # Formata o tempo para ter 5 casas decimais
    tempo_str = f"{res['Tempo (s)']:.5f}"
    print(f"{res['Algoritmo']:<30} | {str(res['Movimentos']):<12} | {tempo_str:<12} | {res['Nós Expandidos']}")

print("="*75)

Iniciando bateria de testes...

Executando Busca em Largura (BFS)...
Executando Busca em Prof. (DFS)...
Executando Aprofundamento Iter. (IDS)...
Executando A* (H1: Peças Fora do Lugar)...
Executando A* (H2: Manhattan Distance)...

                         RESULTADOS DA COMPARAÇÃO                          
Algoritmo                      | Movimentos   | Tempo (s)    | Nós Expandidos
---------------------------------------------------------------------------
Busca em Largura (BFS)         | 4            | 0.00197      | 19
Busca em Prof. (DFS)           | 432          | 0.01142      | 442
Aprofundamento Iter. (IDS)     | 4            | 0.00088      | 68
A* (H1: Peças Fora do Lugar)   | 4            | 0.00015      | 5
A* (H2: Manhattan Distance)    | 4            | 0.00013      | 5
